In [1]:
%pip install minio


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
MINIO_HOST = 'localhost'
MINIO_ACCESS_KEY = 'admin'
MINIO_SECRET_KEY = 'admin1234'

In [3]:
from minio import Minio

minio_client = Minio(
    f"{MINIO_HOST}:9000",
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

In [4]:
minio_client.bucket_exists('user-pics')

False

In [5]:
if not minio_client.bucket_exists('user-pics'):
    minio_client.make_bucket('user-pics')

In [6]:
minio_client.bucket_exists('user-pics')

True

In [7]:
minio_client.list_buckets()

[Bucket('user-pics')]

---

In [9]:
import io

In [10]:
with open('cat.jpg', 'rb') as f:
    data = io.BytesIO(f.read())

result = minio_client.put_object(
    bucket_name='user-pics',
    object_name='cat.jpg',
    data=data,
    length=-1,
    part_size=5*1024*1024
)
print(
    f'created {result.object_name} object; etag: {result.etag}, version-id: {result.version_id}'
)

created cat.jpg object; etag: 9563277ca4f0c8833911b2b70eb3fcec, version-id: None


In [11]:
data.seek(0)

0

In [12]:
result = minio_client.put_object(
    bucket_name='user-pics',
    object_name='cat_meta.jpg',
    data=data,
    length=-1,
    part_size=5*1024*1024,
    metadata={'creationPlace': 'Moscow, Russia'}
)
print(
    f'created {result.object_name} object; etag: {result.etag}, version-id: {result.version_id}'
)

created cat_meta.jpg object; etag: 9563277ca4f0c8833911b2b70eb3fcec, version-id: None


In [13]:
from datetime import datetime, timedelta
from minio.retention import Retention
from minio.commonconfig import GOVERNANCE, Tags

In [14]:
data.seek(0)

0

In [15]:
# date = datetime.utcnow().replace(
#     hour=0, minute=0, second=0, microsecond=0,
# ) + timedelta(days=30)

tags = Tags(for_object=True)
tags['usergroup'] = 'teacher'

result = minio_client.put_object(
    bucket_name='user-pics',
    object_name='cat_tags.jpg',
    data=data,
    length=-1,
    part_size=5*1024*1024,
    metadata={'creationPlace': 'Moscow, Russia'},
    tags=tags,
)

print(
    f'created {result.object_name} object; etag: {result.etag}, version-id: {result.version_id}'
)

created cat_tags.jpg object; etag: 9563277ca4f0c8833911b2b70eb3fcec, version-id: None


---

In [18]:
import json

<img src="http://localhost:9000/user-pics/cat.jpg"></img>

In [ ]:
http://localhost:9000/user-pics/cat.jpg

In [16]:
minio_client.get_bucket_policy('user-pics')

S3Error: S3 operation failed; code: NoSuchBucketPolicy, message: The bucket policy does not exist, resource: /user-pics, request_id: 1865D2E43FAD2BA3, host_id: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8, bucket_name: user-pics

In [19]:
# https://awspolicygen.s3.amazonaws.com/policygen.html

policy = {
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "Statement1",
      "Effect": "Allow",
      "Principal": "*",
      "Action": [
        "s3:GetObject"
      ],
      "Resource": "arn:aws:s3:::user-pics/*"
    }
  ]
}

minio_client.set_bucket_policy('user-pics', json.dumps(policy))

In [20]:
minio_client.get_bucket_policy('user-pics')

'{"Version":"2012-10-17","Statement":[{"Sid":"Statement1","Effect":"Allow","Principal":{"AWS":["*"]},"Action":["s3:GetObject"],"Resource":["arn:aws:s3:::user-pics/*"]}]}'

---

In [21]:
from minio.commonconfig import SnowballObject

In [22]:
data.seek(0)
with open('dog.jpeg', 'rb') as f:
    data_dog = io.BytesIO(f.read())

minio_client.upload_snowball_objects(
    'user-pics',
    [
        SnowballObject('many_dog.jpeg', data=data_dog, length=len(data_dog.getvalue())),
        SnowballObject('many_cat.jpeg', data=data, length=len(data.getvalue()))
    ],
)

---

In [23]:
for i in minio_client.list_objects('user-pics'):
    print(i.object_name)

cat.jpg
cat_meta.jpg
cat_tags.jpg
many_cat.jpeg
many_dog.jpeg


In [24]:
for i in minio_client.list_objects('user-pics'):
    minio_client.remove_object('user-pics', i.object_name)

In [25]:
for i in minio_client.list_objects('user-pics'):
    print(i.object_name)

---

In [26]:
minio_client.remove_bucket('user-pics')

https://min.io/docs/minio/linux/developers/python/API.html